# 가상환경 자동 설정
이 노트북을 실행하면 `.venv` 가상환경 생성 → 패키지 설치(TensorFlow + **PyTorch CUDA**) → Jupyter 커널 등록 → 커널 자동 변경까지 완료됩니다.

- PyTorch는 `https://download.pytorch.org/whl/cu124` 에서 CUDA 12.4 빌드로 설치됩니다.
- NVIDIA 드라이버가 CUDA 12.4 이상을 지원해야 하며, GPU 미탑재 환경이면 `PYTORCH_INDEX_URL`을 `https://download.pytorch.org/whl/cpu` 로 바꾸세요.

In [13]:
import subprocess, sys, os, json, pathlib

# === 설정 ===
VENV_NAME = ".venv"
KERNEL_NAME = "ai-practice-venv"
KERNEL_DISPLAY = "Python (AI-practice)"
REQUIREMENTS_FILE = "requirements.txt"

PROJECT_DIR = pathlib.Path.cwd()
VENV_DIR = PROJECT_DIR / VENV_NAME
REQ_PATH = PROJECT_DIR / REQUIREMENTS_FILE

if os.name == "nt":
    PYTHON = VENV_DIR / "Scripts" / "python.exe"
    PIP = VENV_DIR / "Scripts" / "pip.exe"
else:
    PYTHON = VENV_DIR / "bin" / "python"
    PIP = VENV_DIR / "bin" / "pip"

# requirements.txt 파싱
assert REQ_PATH.exists(), f"{REQUIREMENTS_FILE} 파일을 찾을 수 없습니다: {REQ_PATH}"

PACKAGES = []
PYTORCH_INDEX_URL = None
for line in REQ_PATH.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    if line.startswith("--extra-index-url"):
        PYTORCH_INDEX_URL = line.split(None, 1)[1]
        continue
    if line.startswith("-"):
        continue
    PACKAGES.append(line)

def run(cmd, desc=""):
    if desc:
        print(f"\n>>> {desc}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        lines = result.stdout.strip().split("\n")
        for line in lines[-10:]:
            print(f"  {line}")
    if result.returncode != 0 and result.stderr.strip():
        print(f"  [ERROR] {result.stderr.strip()[:300]}")
    return result.returncode == 0

print(f"프로젝트 경로: {PROJECT_DIR}")
print(f"가상환경 경로: {VENV_DIR}")
print(f"requirements: {REQ_PATH}")
print(f"패키지 목록: {', '.join(PACKAGES)}")
if PYTORCH_INDEX_URL:
    print(f"PyTorch 인덱스: {PYTORCH_INDEX_URL}")

프로젝트 경로: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice
가상환경 경로: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
requirements: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\requirements.txt
패키지 목록: tensorflow, matplotlib, numpy, pandas, scikit-learn, seaborn, ipykernel, torch, torchvision, transformers, datasets, trl, peft, tqdm, rouge-score, unsloth
PyTorch 인덱스: https://download.pytorch.org/whl/cu124


In [14]:
# 1. 가상환경 생성
if VENV_DIR.exists():
    print(f"기존 가상환경 발견: {VENV_DIR}")
else:
    print("가상환경 생성 중...")
    run([sys.executable, "-m", "venv", str(VENV_DIR)], "python -m venv")
    print("가상환경 생성 완료")

# Python 경로 확인
assert PYTHON.exists(), f"Python 실행 파일을 찾을 수 없습니다: {PYTHON}"
print(f"venv Python: {PYTHON}")

기존 가상환경 발견: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
venv Python: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv\Scripts\python.exe


In [15]:
# 2. pip 업그레이드 + requirements.txt로 패키지 설치
run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip"], "pip 업그레이드")
run([str(PIP), "install", "-r", str(REQ_PATH)], f"requirements.txt 패키지 설치")
print("\n패키지 설치 완료")


>>> pip 업그레이드

>>> requirements.txt 패키지 설치

패키지 설치 완료


In [16]:
# 3. Jupyter 커널 등록
run(
    [str(PYTHON), "-m", "ipykernel", "install", "--user",
     f"--name={KERNEL_NAME}", f"--display-name={KERNEL_DISPLAY}"],
    "Jupyter 커널 등록"
)
print(f"\n커널 '{KERNEL_DISPLAY}' ({KERNEL_NAME}) 등록 완료")


>>> Jupyter 커널 등록
  Installed kernelspec ai-practice-venv in C:\Users\PDG\AppData\Roaming\jupyter\kernels\ai-practice-venv

커널 'Python (AI-practice)' (ai-practice-venv) 등록 완료


In [17]:
# 4. 커널/인터프리터 자동 적용
#    (1) 하위 폴더 포함 모든 .ipynb 커널 메타데이터 변경
#    (2) VS Code 워크스페이스 기본 인터프리터를 .venv로 고정 (.py 파일 대응)

# (1) 재귀적으로 모든 노트북 탐색, .venv 내부/체크포인트는 제외
NOTEBOOKS = [
    p for p in PROJECT_DIR.rglob("*.ipynb")
    if VENV_DIR not in p.parents and ".ipynb_checkpoints" not in p.parts
]

changed = []
for nb_path in NOTEBOOKS:
    if nb_path.name == "envSetting.ipynb":
        continue
    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            nb = json.load(f)
    except Exception as e:
        print(f"  [!] {nb_path.relative_to(PROJECT_DIR)} 읽기 실패: {e}")
        continue

    kernel_info = nb.get("metadata", {}).get("kernelspec", {})
    if kernel_info.get("name") == KERNEL_NAME:
        print(f"  {nb_path.relative_to(PROJECT_DIR)} - 이미 설정됨, 건너뜀")
        continue

    nb.setdefault("metadata", {})["kernelspec"] = {
        "display_name": KERNEL_DISPLAY,
        "language": "python",
        "name": KERNEL_NAME,
    }
    with open(nb_path, "w", encoding="utf-8") as f:
        json.dump(nb, f, ensure_ascii=False, indent=1)
    changed.append(str(nb_path.relative_to(PROJECT_DIR)))
    print(f"  {nb_path.relative_to(PROJECT_DIR)} -> 커널 변경 완료")

print(f"\n총 {len(changed)}개 노트북 커널 변경 완료")
if changed:
    print(f"변경된 파일: {', '.join(changed)}")

# (2) VS Code 워크스페이스 인터프리터 고정 — 이 폴더 안의 모든 .py는 venv로 실행됨
vscode_dir = PROJECT_DIR / ".vscode"
vscode_dir.mkdir(exist_ok=True)
settings_path = vscode_dir / "settings.json"

settings = {}
if settings_path.exists():
    try:
        settings = json.loads(settings_path.read_text(encoding="utf-8"))
    except Exception:
        print(f"  [!] 기존 settings.json 파싱 실패, 새로 만듭니다.")
        settings = {}

settings["python.defaultInterpreterPath"] = str(PYTHON)
# 터미널 열 때 venv 자동 활성화
settings["python.terminal.activateEnvironment"] = True

settings_path.write_text(
    json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"\n.vscode/settings.json 갱신 완료")
print(f"  python.defaultInterpreterPath = {PYTHON}")
print("\n노트북/스크립트를 다시 열면 .venv 환경이 자동 적용됩니다.")

  basic\01_numpy.ipynb - 이미 설정됨, 건너뜀
  basic\02_pandas.ipynb - 이미 설정됨, 건너뜀
  basic\03_matplotlib.ipynb - 이미 설정됨, 건너뜀
  basic\04_scikit_learn.ipynb - 이미 설정됨, 건너뜀
  basic\05_tensorFlow.ipynb - 이미 설정됨, 건너뜀
  basic\06_pyTorch.ipynb - 이미 설정됨, 건너뜀

총 0개 노트북 커널 변경 완료

.vscode/settings.json 갱신 완료
  python.defaultInterpreterPath = c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv\Scripts\python.exe

노트북/스크립트를 다시 열면 .venv 환경이 자동 적용됩니다.


In [18]:
# 5. 설치 확인 (python -m pip show 방식 — 안정적 패키지 확인)
print("=== 설치 확인 ===")
for pkg in PACKAGES:
    result = subprocess.run(
        [str(PYTHON), "-m", "pip", "show", pkg],
        capture_output=True, text=True, encoding="utf-8"
    )
    if result.returncode == 0 and result.stdout:
        for line in result.stdout.splitlines():
            if line.startswith("Name:") or line.startswith("Version:"):
                print(f"  {line.strip()}", end="  ")
        print()
    else:
        print(f"  [!] {pkg} - 설치 안 됨")

# PyTorch CUDA 동작 확인
print("\n=== PyTorch CUDA 확인 ===")
check = subprocess.run(
    [str(PYTHON), "-c",
     "import torch; "
     "print('torch:', torch.__version__); "
     "print('cuda available:', torch.cuda.is_available()); "
     "print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"],
    capture_output=True, text=True, encoding="utf-8"
)
print(check.stdout.strip() if check.stdout else check.stderr.strip() if check.stderr else "PyTorch 미설치")

print(f"\n=== 완료 ===")
print(f"가상환경: {VENV_DIR}")
print(f"커널: {KERNEL_DISPLAY}")
print(f"패키지 관리: {REQ_PATH}")
print("노트북을 닫고 다시 열면 가상환경 커널로 실행됩니다.")

=== 설치 확인 ===
  Name: tensorflow    Version: 2.21.0  
  Name: matplotlib    Version: 3.10.8  
  Name: numpy    Version: 2.2.6  
  Name: pandas    Version: 2.3.3  
  Name: scikit-learn    Version: 1.7.2  
  Name: seaborn    Version: 0.13.2  
  Name: ipykernel    Version: 7.2.0  
  Name: torch    Version: 2.10.0+cu128  
  Name: torchvision    Version: 0.25.0+cu128  
  Name: transformers    Version: 5.5.0  
  Name: datasets    Version: 4.3.0  
  Name: trl    Version: 0.24.0  
  Name: peft    Version: 0.19.0  
  Name: tqdm    Version: 4.67.3  
  Name: rouge-score    Version: 0.1.2  
  Name: unsloth    Version: 2026.4.5  

=== PyTorch CUDA 확인 ===
torch: 2.10.0+cu128
cuda available: True
device: NVIDIA GeForce RTX 3060 Laptop GPU

=== 완료 ===
가상환경: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
커널: Python (AI-practice)
패키지 관리: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\requirements.txt
노트북을 닫고 다시 열면 가상환경 커널로 실행됩니다.
